In [1]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

Found existing installation: keras 3.10.0
Uninstalling keras-3.10.0:
  Successfully uninstalled keras-3.10.0
Found existing installation: matplotlib 3.10.0
Uninstalling matplotlib-3.10.0:
  Successfully uninstalled matplotlib-3.10.0
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.simplefilter('ignore')

In [3]:
import os
import sys
import subprocess

In [4]:
def set_env(input_archive, temp_dir):

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        f'{temp_dir}/wheels', 
        'unsloth', 
        'trl', 
        'vllm', 
        'openai_harmony'
    ], check=True)

In [5]:
set_env(
    input_archive='/kaggle/input/aimo-3-utils/wheels.tar.gz', 
    temp_dir='/kaggle/tmp/setup'
)

Looking in links: /kaggle/tmp/setup/wheels
Processing /kaggle/tmp/setup/wheels/unsloth-2025.12.9-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/trl-0.24.0-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/vllm-0.11.2-cp38-abi3-manylinux1_x86_64.whl
Processing /kaggle/tmp/setup/wheels/openai_harmony-0.0.8-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/tmp/setup/wheels/unsloth_zoo-2025.12.7-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/tyro-1.0.3-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/xformers-0.0.33.post1-cp39-abi3-manylinux_2_28_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/bitsandbytes-0.49.0-py3-none-manylinux_2_24_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/datasets-4.3.0-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/prometheus_fastapi_instrumentator-7.1.0-py3-none-any.whl (from vllm)
Processing /kaggle/tmp/setup/wheels/lm_format_enforcer-0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kauldron 1.3.0 requires scikit-learn, which is not installed.
kauldron 1.3.0 requires tensorflow, which is not installed.
ydata-profiling 4.18.0 requires matplotlib<=3.10,>=3.5, which is not installed.
pyldavis 3.4.1 requires scikit-learn>=1.0.0, which is not installed.
stable-baselines3 2.1.0 requires matplotlib, which is not installed.
sentence-transformers 5.1.1 requires scikit-learn, which is not installed.
librosa 0.11.0 requires scikit-learn>=1.1.0, which is not installed.
cuml-cu12 25.6.0 requires scikit-learn>=1.5, which is not installed.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
bigframes 2.26.0 requires matplotlib>=3.7.1, which is not installed.
arviz 0.22.0 requires matplotlib>=3.8, which is not installed.
pynndescent 0.5.13 requires scikit-learn>=0.

In [6]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])

cl100k_base.tiktoken
o200k_base.tiktoken


CompletedProcess(args=['ls', '/kaggle/tmp/setup/tiktoken_encodings'], returncode=0)

In [7]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

In [8]:
import gc
import re
import math
import time
import copy
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, 
    load_harmony_encoding, 
    SystemContent, 
    ReasoningEffort, 
    ToolNamespaceConfig, 
    Author, 
    Message, 
    Role, 
    TextContent, 
    Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

In [9]:
class ReasoningDatasetManager:
    """Manages saving successful reasoning traces to CSV in Harmony format."""
    
    def __init__(self, output_path: str, input_path: str = None):
        self.output_path = output_path
        self.input_path = input_path
        self._lock = threading.Lock()
        self._buffer = []
        
        # Load existing data if available from input path
        self._existing_df = None
        if input_path and os.path.exists(input_path):
            try:
                self._existing_df = pd.read_csv(input_path)
            except Exception:
                self._existing_df = None
    
    def serialize_conversation(self, conversation: Conversation, encoding) -> str:
        """Serialize a Conversation object to Harmony format string."""
        tokens = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
        # Decode tokens back to string representation
        harmony_text = encoding.decode(tokens)
        return harmony_text
    
    def add_successful_attempt(
        self, 
        problem: str, 
        conversation: Conversation, 
        answer: int | str,
        ground_truth: int | None,
        reasoning_effort: str,
        encoding
    ):
        """Add a successful attempt to the buffer."""
        reasoning = self.serialize_conversation(conversation, encoding)
        
        with self._lock:
            self._buffer.append({
                'problem': problem,
                'reasoning': reasoning,
                'answer': ground_truth,  # Ground truth answer (can be None)
                'predicted_answer': answer,  # Model's predicted answer
                'reasoning_effort': reasoning_effort
            })
    
    def flush_to_csv(self):
        """Write buffered data to CSV file."""
        with self._lock:
            if not self._buffer:
                return
            
            new_df = pd.DataFrame(self._buffer)
            
            # Combine with existing data if available
            if self._existing_df is not None:
                combined_df = pd.concat([self._existing_df, new_df], ignore_index=True)
            else:
                # Check if output file already exists
                if os.path.exists(self.output_path):
                    try:
                        existing_output_df = pd.read_csv(self.output_path)
                        combined_df = pd.concat([existing_output_df, new_df], ignore_index=True)
                    except Exception:
                        combined_df = new_df
                else:
                    combined_df = new_df
            
            # Ensure output directory exists
            os.makedirs(os.path.dirname(self.output_path), exist_ok=True)
            
            combined_df.to_csv(self.output_path, index=False)
            self._buffer.clear()
            self._existing_df = combined_df  # Update for subsequent flushes
    
    def get_buffer_size(self) -> int:
        """Return current buffer size."""
        with self._lock:
            return len(self._buffer)

In [10]:
class CFG:

    system_prompt = (
        'You are a world-class International Mathematical Olympiad (IMO) competitor. '
        # 'The final answer must be a non-negative integer between 0 and 99999. '
        'You must place the final integer answer inside \\boxed{}.'
    )
    
    tool_prompt = (
        'Use this tool to execute Python code. '
        'The environment is a stateful Jupyter notebook. '
        'You must use print() to output results.'
    )

    preference_prompt = (
        'Use `math`, `numpy`,`sympy`,`itertools` and `collections` to solve the problem.'
    )

    # Reasoning effort: 'high', 'medium', or 'low'
    reasoning_effort = 'low'

    served_model_name = 'gpt-oss'
    model_path = '/kaggle/input/gpt-oss-120b/transformers/default/1'
    
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    high_problem_timeout = 900
    base_problem_timeout = 300

    notebook_limit = 17640
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 10
    sandbox_timeout = 5

    stream_interval = 200
    context_tokens = 65536
    search_tokens = 1024
    buffer_tokens = 512
    batch_size = 256
    early_stop = 4
    attempts = 8
    workers = 16
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.96
    temperature = 1.0
    min_p = 0.02

    # Dataset output settings
    dataset_output_path = '/kaggle/working/reasoning_dataset.csv'
    dataset_input_path = '/kaggle/input/reasoning_dataset.csv'  # For appending

In [11]:
set_seed(CFG.seed)

In [12]:
class AIMO3Template:

    def __init__(self, reasoning_effort: str = 'high'):
        self.reasoning_effort = reasoning_effort.lower()

    def _get_reasoning_effort_enum(self) -> ReasoningEffort:
        mapping = {
            'high': ReasoningEffort.HIGH,
            'medium': ReasoningEffort.MEDIUM,
            'low': ReasoningEffort.LOW
        }
        return mapping.get(self.reasoning_effort, ReasoningEffort.HIGH)

    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:

        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=self._get_reasoning_effort_enum())
            .with_tools(tool_config)
        )

    def apply_chat_template(
        self, 
        system_prompt: str, 
        user_prompt: str, 
        tool_config: ToolNamespaceConfig
    ) -> list[Message]:

        system_content = self.get_system_content(system_prompt, tool_config)        
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)

        user_message = Message.from_role_and_content(Role.USER, user_prompt)

        return [system_message, user_message]

In [13]:
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import numpy as np\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):

        self.execute('%reset -f')
        self.execute('import gc; gc.collect()')

        self.execute(
            'import math\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import numpy as np\n'
        )

    def __del__(self):

        self.close()

In [14]:
class AIMO3Tool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):

        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        
        self._owns_session = sandbox is None
        
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):

        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:

        lines = code.strip().split('\n')

        if not lines:
            return code

        last_line = lines[-1].strip()

        if 'print' in last_line or 'import' in last_line:
            return code

        if not last_line:
            return code

        if last_line.startswith('#'):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    @property
    def instruction(self) -> str:

        return self._tool_prompt

    @property
    def tool_config(self) -> ToolNamespaceConfig:

        return ToolNamespaceConfig(
            name='python', 
            description=self.instruction, 
            tools=[]
        )

    def _make_response(self, output: str, channel: str | None = None) -> Message:

        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')

        if channel:
            message = message.with_channel(channel)

        return message

    def process_sync_plus(self, message: Message) -> list[Message]:

        self._ensure_session()
        raw_script = message.content[0].text
        final_script = self._ensure_last_print(raw_script)

        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)

            except TimeoutError as exc:
                output = f'[ERROR] {exc}'

        return [self._make_response(output, channel=message.channel)]

    def close(self):

        if self._jupyter_session is not None:
            if self._owns_session:
                self._jupyter_session.close()

            self._jupyter_session = None

    def __del__(self):

        self.close()

In [15]:
class AIMO3Solver:

    def __init__(self, cfg, port: int = 8000):

        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template(reasoning_effort=cfg.reasoning_effort)
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()
        
        # Initialize dataset manager for saving reasoning traces
        self.dataset_manager = ReasoningDatasetManager(
            output_path=cfg.dataset_output_path,
            input_path=cfg.dataset_input_path
        )
        self.current_ground_truth = None  # Will be set per problem

        self._preload_model_weights()
        
        self.server_process = self._start_server()

        self.client = OpenAI(
            base_url=self.base_url, 
            api_key=self.api_key, 
            timeout=self.cfg.session_timeout
        )

        self._wait_for_server()
        self._initialize_kernels()

        self.notebook_start_time = time.time()
        self.problems_remaining = 50

    def _preload_model_weights(self) -> None:

        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        
        files_to_load = []
        total_size = 0

        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)

                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)

        def _read_file(path: str) -> None:

            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))

        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

    def _start_server(self) -> subprocess.Popen:

        cmd = [
            sys.executable, 
            '-m', 
            'vllm.entrypoints.openai.api_server', 
            '--seed', 
            str(self.cfg.seed), 
            '--model', 
            self.cfg.model_path, 
            '--served-model-name', 
            self.cfg.served_model_name, 
            '--tensor-parallel-size', 
            '1', 
            '--max-num-seqs', 
            str(self.cfg.batch_size), 
            '--gpu-memory-utilization', 
            str(self.cfg.gpu_memory_utilization), 
            '--host', 
            '0.0.0.0', 
            '--port', 
            str(self.port), 
            '--dtype', 
            self.cfg.dtype, 
            '--kv-cache-dtype', 
            self.cfg.kv_cache_dtype, 
            '--max-model-len', 
            str(self.cfg.context_tokens), 
            '--stream-interval', 
            str(self.cfg.stream_interval), 
            '--async-scheduling', 
            '--enable-prefix-caching'
        ]

        self.log_file = open('vllm_server.log', 'w')

        return subprocess.Popen(
            cmd, 
            stdout=self.log_file, 
            stderr=subprocess.STDOUT, 
            start_new_session=True
        )

    def _wait_for_server(self):

        print('Waiting for vLLM server...')
        start_time = time.time()

        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()

            if return_code is not None:
                self.log_file.flush()

                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()

                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')

            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')

                return

            except Exception:
                time.sleep(1)

        raise RuntimeError('Server failed to start (timeout).\n')

    def _initialize_kernels(self) -> None:

        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()

        self.sandbox_pool = queue.Queue()

        def _create_sandbox():
            
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]

            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())

        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')

    def _scan_for_answer(self, text: str) -> int | None:

        pattern = r'\\boxed\s*\{\s*([0-9,]+)\s*\}'
        matches = re.findall(pattern, text)

        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                # value = int(clean_value)

                # if 0 <= value <= 99999:
                #     return value
                return clean_value

            except ValueError:
                pass

        return None

    def _process_attempt(
        self, 
        problem: str, 
        system_prompt: str, 
        attempt_index: int, 
        stop_event: threading.Event, 
        deadline: float
    ) -> dict:

        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1, 
                'Answer': None, 
                'Python Calls': 0, 
                'Python Errors': 0, 
                'Response Length': 0,
                'Conversation': None
            }

        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None
        final_conversation = None

        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))

        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)

            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout, 
                tool_prompt=self.cfg.tool_prompt, 
                sandbox=sandbox
            )

            encoding = self.encoding
            messages = self.template.apply_chat_template(
                system_prompt, 
                problem, 
                local_tool.tool_config
            )

            conversation = Conversation.from_messages(messages)

            for _ in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    break

                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)

                if max_tokens < self.cfg.buffer_tokens:
                    break

                # Create stream request with error handling
                stream = None
                try:
                    stream = self.client.completions.create(
                        model=self.cfg.served_model_name, 
                        temperature=self.cfg.temperature, 
                        max_tokens=max_tokens, 
                        prompt=prompt_ids, 
                        seed=attempt_seed, 
                        stream=True, 
                        extra_body={
                            'min_p': self.cfg.min_p, 
                            'stop_token_ids': self.stop_token_ids, 
                            'return_token_ids': True
                        },
                        timeout=max(0,deadline - time.time()),
                    )
                except Exception as e:
                    print(f"⚠️ Failed to create completion stream: {e}")
                    # Break this iteration and try next one
                    break
    
                if stream is None:
                    # Stream creation failed, skip this iteration
                    continue

                try:
                    token_buffer = []
                    text_chunks = []

                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break

                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text

                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            total_tokens += len(new_tokens)
                            text_chunks.append(new_text)

                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer = self._scan_for_answer(search_text)

                            if answer is not None:
                                final_answer = answer
                                # Parse and add remaining tokens to conversation before breaking
                                if token_buffer:
                                    new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                                    conversation.messages.extend(new_messages)
                                final_conversation = copy.deepcopy(conversation)
                                break

                finally:
                    stream.close()

                if final_answer is not None:
                    break

                if not token_buffer:
                    break

                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]

                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    final_answer = self._scan_for_answer(answer_text)
                    if final_answer is not None:
                        final_conversation = copy.deepcopy(conversation)
                    break

                if last_message.recipient == 'python':
                    python_calls += 1
                    print("🐍 Executing Python code...")
                    tool_responses = local_tool.process_sync_plus(last_message)

                    response_text = tool_responses[0].content[0].text

                    if response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text:
                        python_errors += 1

                    conversation.messages.extend(tool_responses)

        except Exception as exc:
            python_errors += 1

        finally:
            if local_tool is not None:
                local_tool.close()

            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)

        return {
            'Attempt': attempt_index + 1, 
            'Response Length': total_tokens, 
            'Python Calls': python_calls, 
            'Python Errors': python_errors, 
            'Answer': final_answer,
            'Conversation': final_conversation
        }

    def _select_answer(self, detailed_results: list) -> int:

        stats = defaultdict(lambda: {'votes': 0, 'calls': 0})

        for result in detailed_results:
            answer = result['Answer']

            if answer is not None:
                stats[answer]['votes'] += 1
                stats[answer]['calls'] += result['Python Calls']

        sorted_stats = sorted(
            stats.items(), 
            key=lambda item: (item[1]['votes'], item[1]['calls']), 
            reverse=True
        )

        vote_data = []

        for answer, data in sorted_stats:
            vote_data.append((answer, data['votes'], data['calls']))

        vote_dataframe = pd.DataFrame(vote_data, columns=['Answer', 'Votes', 'Calls'])
        display(vote_dataframe)

        final_answer = sorted_stats[0][0]
        final_votes = sorted_stats[0][1]['votes']
        final_calls = sorted_stats[0][1]['calls']

        print(f'\nFinal Result: {final_answer} | Votes: {final_votes} | Calls: {final_calls}\n')

        return final_answer

    def solve_problem(self, problem: str, ground_truth: int | None = None) -> int:
        
        problem_start_time = time.time()
        print(f'\nProblem: {problem}\n')
        
        # Store ground truth for this problem
        self.current_ground_truth = ground_truth

        user_input = f'{problem} {self.cfg.preference_prompt}'
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout

        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)

        deadline = time.time() + budget

        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f}\n')

        tasks = []

        for attempt_index in range(self.cfg.attempts):
            tasks.append((self.cfg.system_prompt, attempt_index))

        detailed_results = []
        valid_answers = []

        stop_event = threading.Event()

        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)

        try:
            futures = []

            for (system_prompt, attempt_index) in tasks:
                future = executor.submit(
                    self._process_attempt, 
                    user_input, 
                    system_prompt, 
                    attempt_index, 
                    stop_event, 
                    deadline
                )

                futures.append(future)

            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)

                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])

                    counts = Counter(valid_answers).most_common(1)

                    if counts and counts[0][1] >= self.cfg.early_stop:
                        stop_event.set()

                        for f in futures:
                            f.cancel()

                        break

                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue

        finally:
            executor.shutdown(wait=False, cancel_futures=True)
            self.problems_remaining = max(0, self.problems_remaining - 1)

        # print the inference time and budget
        used_time = time.time() - problem_start_time
        saved_time = max(0.0, budget - used_time)
        print(f"[Budget]: {budget:.2f}s\n")
        print(f"[inference] Took {used_time:.2f}s\n")
        print(f"[Saved time]: {saved_time:.2f}s\n")

        # Save successful attempts to dataset (only ones with valid answers and conversations)
        for result in detailed_results:
            if result['Answer'] is not None and result['Conversation'] is not None:
                self.dataset_manager.add_successful_attempt(
                    problem=problem,
                    conversation=result['Conversation'],
                    answer=result['Answer'],
                    ground_truth=self.current_ground_truth,
                    reasoning_effort=self.cfg.reasoning_effort,
                    encoding=self.encoding
                )
        
        # Flush dataset to CSV after each problem
        self.dataset_manager.flush_to_csv()

        if detailed_results:
            # Create display dataframe without Conversation column
            display_results = [{k: v for k, v in r.items() if k != 'Conversation'} for r in detailed_results]
            results_dataframe = pd.DataFrame(display_results)
            # results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64')
            display(results_dataframe)

        if not valid_answers:
            print('\nResult: 0\n')

            return 0

        return self._select_answer(detailed_results)

    def __del__(self):
        
        # Flush any remaining dataset entries
        if hasattr(self, 'dataset_manager'):
            self.dataset_manager.flush_to_csv()

        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()

        if hasattr(self, 'log_file'):
            self.log_file.close()

        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()

                except Exception:
                    pass

In [16]:
solver = AIMO3Solver(CFG)

Loading model weights from /kaggle/input/gpt-oss-120b/transformers/default/1 into OS Page Cache...
Processed 26 files (65.28 GB) in 64.21 seconds.

Waiting for vLLM server...
Server is ready (took 128.46 seconds).

Initializing 16 persistent Jupyter kernels...
Kernels initialized in 3.07 seconds.



In [17]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    global correct_count, total_count, predictions
    
    question_id = id_.item(0)
    question_text = question.item(0)
    
    print("------")
    print(f"ID: {question_id}")
    print(f"Question: {question_text[:200]}...")
    
    # Get ground truth if available
    gt = ground_truth.get(question_id, None)
    
    final_answer = solver.solve_problem(question_text, ground_truth=gt)
    predictions[question_id] = final_answer

    # Check accuracy if ground truth available
    total_count += 1
    if gt is not None:
        is_correct = (final_answer == gt)
        if is_correct:
            correct_count += 1
        status = "✅" if is_correct else "❌"
        print(f"Answer: {final_answer} | Ground Truth: {gt} | {status}")
        print(f"📊 Running Accuracy: {correct_count}/{total_count} ({100*correct_count/total_count:.1f}%)")
    else:
        print(f"Answer: {final_answer}")
    
    print("------\n")
    
    return pl.DataFrame({'id': question_id, 'answer': final_answer})

In [18]:
# ============================================================================
# Dataset Configuration - Modify this section for different datasets
# ============================================================================
# Input CSV path
INPUT_CSV_PATH = "/kaggle/input/olymmath-hard/OlymMATH-EN-HARD.csv"

# Column mapping - adjust these based on your dataset's column names
# Set to None if column doesn't exist
ID_COLUMN = "id"           # Column containing unique problem ID (or None to auto-generate)
QUESTION_COLUMN = "problem" if "problem" in pd.read_csv(INPUT_CSV_PATH, nrows=0).columns else "question"
ANSWER_COLUMN = "answer"   # Column containing ground truth answer (or None if not available)

# ============================================================================

# Load reference data
df = pd.read_csv(INPUT_CSV_PATH)

# Handle column normalization
# If no ID column, create one
if ID_COLUMN is None or ID_COLUMN not in df.columns:
    df["id"] = range(len(df))
    ID_COLUMN = "id"
else:
    df = df.rename(columns={ID_COLUMN: "id"})

# Normalize question column
if QUESTION_COLUMN != "question" and QUESTION_COLUMN in df.columns:
    df = df.rename(columns={QUESTION_COLUMN: "question"})
elif "question" not in df.columns:
    # Try common alternatives
    for alt in ["problem", "text", "prompt", "query"]:
        if alt in df.columns:
            df = df.rename(columns={alt: "question"})
            break

# Store ground truth answers for accuracy calculation (only in local mode)
if ANSWER_COLUMN and ANSWER_COLUMN in df.columns:
    ground_truth = dict(zip(df["id"], df[ANSWER_COLUMN]))
elif "answer" in df.columns:
    ground_truth = dict(zip(df["id"], df["answer"]))
else:
    ground_truth = {}

# Create input file for inference (id and question only)
reference_df = df[["id", "question"]].copy()
reference_df.to_csv("reference.csv", index=False)

# Track predictions for accuracy calculation
predictions = {}
correct_count = 0
total_count = 0

print(f"Loaded {len(df)} problems from {INPUT_CSV_PATH}")
print(f"Ground truth available: {len(ground_truth) > 0}")

Loaded 100 problems from /kaggle/input/olymmath-hard/OlymMATH-EN-HARD.csv
Ground truth available: True


In [19]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
    
else:
    inference_server.run_local_gateway(("reference.csv",))
    #inference_server.run_local_gateway(
    #    ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',)
    #)

------
ID: 98
Question: Let $a_1, a_2, \cdots, a_{20}$ be $20$ distinct positive integers, and the set $\{a_i + a_j | 1 \le i, j \le 20\}$ contains $201$ distinct elements. Find the minimum possible number of distinct elemen...

Problem: Let $a_1, a_2, \cdots, a_{20}$ be $20$ distinct positive integers, and the set $\{a_i + a_j | 1 \le i, j \le 20\}$ contains $201$ distinct elements. Find the minimum possible number of distinct elements in the set $\{|a_i - a_j| | 1 \le i, j \le 20\}$.

Budget: 900.00 seconds | Deadline: 1768478032.57

🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
[Budget]: 900.00s

[inference] Took 19.47s

[Saved time]: 880.53s



,Attempt,Response Length,Python Calls,Python Errors,Answer
0,2,583,0,0,19
1,5,595,0,0,191
2,8,817,1,0,101
3,7,827,1,0,19
4,1,783,1,0,39
5,4,813,2,0,20
6,3,878,3,0,165
7,6,995,2,2,20


,Answer,Votes,Calls
0,20,2,4
1,19,2,1
2,165,1,3
3,101,1,1
4,39,1,1
5,191,1,0



Final Result: 20 | Votes: 2 | Calls: 4

Answer: 20 | Ground Truth: 100 | ❌
📊 Running Accuracy: 0/1 (0.0%)
------

------
ID: 49
Question: Given the set of integers $A = \{1, 2, \cdots, 100\}$. Let the function $f: A \rightarrow A$ satisfy: (1) for any $1 \leqslant i \leqslant 99$, we have $|f(i) - f(i+1)| \leqslant 1$; (2) for any $1 \l...

Problem: Given the set of integers $A = \{1, 2, \cdots, 100\}$. Let the function $f: A \rightarrow A$ satisfy: (1) for any $1 \leqslant i \leqslant 99$, we have $|f(i) - f(i+1)| \leqslant 1$; (2) for any $1 \leqslant i \leqslant 100$, we have $f(f(i)) = 100$. Find the minimum possible value of $\sum_{i=1}^{100} f(i)$.

Budget: 900.00 seconds | Deadline: 1768478052.12

🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
[Budget]: 900.00s

[inference] Took 27.12s

[Saved time]: 872.88s



,Attempt,Response Length,Python Calls,Python Errors,Answer
0,8,1409,0,0,9902
1,7,1926,0,0,9902
2,4,1939,0,0,10000
3,5,2042,0,0,10000
4,2,2076,1,0,8350
5,6,2374,2,0,9902
6,1,1882,1,1,10000
7,3,3634,4,0,5050


...

,Answer,Votes,Calls
0,3,4,12



Final Result: 3 | Votes: 4 | Calls: 12

Answer: 3 | Ground Truth: 15 | ❌
📊 Running Accuracy: 19/99 (19.2%)
------

------
ID: 27
Question: Given an ellipse $C: x^{2} / a^{2}+y^{2} / b^{2}=1$ $(a>b>0)$ with eccentricity $e=4 / 5$, let $P$ be any point on the ellipse different from the left and right vertices $A$ and $B$ on the major axis,...

Problem: Given an ellipse $C: x^{2} / a^{2}+y^{2} / b^{2}=1$ $(a>b>0)$ with eccentricity $e=4 / 5$, let $P$ be any point on the ellipse different from the left and right vertices $A$ and $B$ on the major axis, $F_{1}$ and $F_{2}$ are the left and right foci of the ellipse respectively, and $\angle APB=2 \alpha$, $\angle F_{1} P F_{2}=2 \beta$. Find the minimum value of $\tan \beta \cdot \tan 2 \alpha$.

Budget: 900.00 seconds | Deadline: 1768479909.40

🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executing Python code...
🐍 Executi

,Attempt,Response Length,Python Calls,Python Errors,Answer
0,7,1062,4,0,None
1,8,1317,4,0,None
2,1,1577,5,0,None
3,3,1616,2,0,None
4,6,1585,9,0,None
5,4,2110,2,0,None
6,2,2159,2,0,5
7,5,3209,5,1,5


,Answer,Votes,Calls
0,5,2,7



Final Result: 5 | Votes: 2 | Calls: 7

Answer: 5 | Ground Truth: -\frac{5}{2} | ❌
📊 Running Accuracy: 19/100 (19.0%)
------

